# Purpose
Perform an empirical trajectory reproducibility and determinism validation experiment for a VizDoom recording. Test whether **seed + action replay** is sufficient to perfectly reconstruct the original recorded trajectory.


# Experimental definition
For the recorded episode, we will test:
`same environment configuration + same initial seed + same ordered action sequence -> same trajectory?`

Every replay will reset using the exact recorded episode seed.
We will feed the complete recorded action sequence back into the environment in order, and compare the generated variables (reward, terminated, truncated, game variables) to the originally recorded sequence. 
We will do this `N_REPLAYS = 100` times.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import gymnasium as gym
import vizdoom.gymnasium_wrapper
from IPython.display import display

sys.path.append(os.path.abspath("../"))

N_REPLAYS = 100

# To test the other dataset, swap the comments below and RESTART YOUR KERNEL!
DATASET_DIR = os.path.abspath("../data/sub-01_20260909-151927") # Defend Center
# DATASET_DIR = os.path.abspath("../data/sub-01_20260909-135403") # Deadly Corridor



# Locate and inspect recordings


In [ ]:
def inspect_dataset(data_dir):
    print(f"--- Inspecting {os.path.basename(data_dir)} ---")
    manifest_path = os.path.join(data_dir, "manifest.json")
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    
    game_curriculum = next(p for p in manifest["curriculum"] if p.get("type") == "game")
    game_phase = next(p for p in manifest["phases"] if p.get("type") == "game")
    npz_name = game_phase["data_file"]
    npz_path = os.path.join(data_dir, npz_name)
    data = np.load(npz_path, allow_pickle=True)
    
    print(f"Game: {game_curriculum['game']} (Backend: {game_curriculum['backend']})")
    print(f"Total recorded steps: {len(data['actions'])}")
    print(f"Episode seeds: {data['episode_seeds']}")
    print("\nArrays saved in NPZ:")
    for key in data.files:
        arr = data[key]
        if hasattr(arr, 'shape'):
            print(f"  {key:15s} | shape: {str(arr.shape):14s} | dtype: {arr.dtype}")
        else:
            print(f"  {key:15s} | {type(arr)}")
    print(f"Env kwargs: {game_curriculum.get('env_kwargs', {})}\n")
    return manifest, data, game_curriculum

manifest, data, phase = inspect_dataset(DATASET_DIR)



# Reconstruct exact VizDoom configuration


In [ ]:
env_kwargs = phase.get("env_kwargs", {})
game_name = phase["game"]


In [ ]:
actions = data["actions"]


# Replay implementation


In [ ]:
def run_replays(game_name, env_kwargs, seed, ref_data, n_replays):
    ref_actions = ref_data["actions"]
    ref_rewards = ref_data["rewards"]
    ref_terms = ref_data["terminated"]
    ref_truncs = ref_data["truncated"]
    ref_gamevars = ref_data["gamevariables"]
    
    # NEW: extract episode IDs and seeds to handle datasets with multiple episodes
    ref_episode_ids = ref_data.get("episode_id", np.zeros(len(ref_actions), dtype=int))
    ref_episode_seeds = ref_data["episode_seeds"]
    
    T = len(ref_actions)
    results = []
    
    print(f"Starting {n_replays} replays of {game_name} ({T} steps each)...")
    env = gym.make(game_name, render_mode="rgb_array", **env_kwargs)
    
    for r in range(n_replays):
        current_ep = -1
        diverged_step = None
        diverged_vars = []
        
        for t in range(T):
            ep_id = int(ref_episode_ids[t])
            # If the recorded step belongs to a new episode, reset the environment
            if ep_id != current_ep:
                obs, info = env.reset(seed=int(ref_episode_seeds[ep_id]))
                current_ep = ep_id
                
            obs, reward, term, trunc, info = env.step(ref_actions[t])
            
            if isinstance(obs, dict) and "gamevariables" in obs:
                gvars = np.asarray(obs["gamevariables"])
            else:
                state = env.unwrapped.game.get_state()
                if state is not None and state.game_variables is not None:
                    gvars = state.game_variables.copy()
                else:
                    gvars = np.zeros_like(ref_gamevars[t])
            
            mismatches = []
            if not np.isclose(reward, ref_rewards[t], atol=1e-5): mismatches.append("reward")
            if bool(term) != bool(ref_terms[t]): mismatches.append("terminated")
            if bool(trunc) != bool(ref_truncs[t]): mismatches.append("truncated")
            if not np.allclose(gvars, ref_gamevars[t], atol=1e-5): mismatches.append("gamevariables")
            
            if len(mismatches) > 0 and diverged_step is None:
                diverged_step = t
                diverged_vars = mismatches
                break
                
        exact = (diverged_step is None)
        results.append({
            "replay_idx": r, "exact": exact, "diverged_step": diverged_step,
            "diverged_vars": diverged_vars, "reward_match": exact or ("reward" not in diverged_vars),
            "gamevars_match": exact or ("gamevariables" not in diverged_vars),
            "term_match": exact or ("terminated" not in diverged_vars),
            "trunc_match": exact or ("truncated" not in diverged_vars)
        })
        
    env.close()
    return pd.DataFrame(results)



# Run experiment


In [ ]:
df = run_replays(game_name, env_kwargs, None, data, N_REPLAYS)
print(f"Dataset ({game_name}): {df['exact'].sum()} / {N_REPLAYS} fully matched.")


# Reproducibility figures


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

total_steps = len(actions)
total_states = len(df) * total_steps
perfect_runs = df['exact'].sum()
divergent_runs = len(df) - perfect_runs
match_rate = (perfect_runs / len(df)) * 100

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis("off")

table_data = [
    ["Total Replays Executed", f"{len(df)}"],
    ["Total Timesteps / Run", f"{total_steps:,}"],
    ["Total States Verified", f"{total_states:,}"],
    ["Perfect Trajectories", f"{perfect_runs}"],
    ["Divergent Trajectories", f"{divergent_runs}"],
    ["Trajectory Match Rate", f"{match_rate:.1f}%"]
]

col_labels = ["Metric", "Result"]

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    colColours=["#2b5c8f", "#2b5c8f"]
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.0, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(color="white", weight="bold")
    elif row > 0:
        # Highlight logic for the "Result" column
        if col == 1:
            if "Perfect Trajectories" in table_data[row-1][0] and perfect_runs == len(df):
                cell.set_facecolor("#e8f5e9") # Green
            elif "Divergent" in table_data[row-1][0] and divergent_runs == 0:
                cell.set_facecolor("#e8f5e9") # Green
            elif "Divergent" in table_data[row-1][0] and divergent_runs > 0:
                cell.set_facecolor("#ffebee") # Red
            elif "Match Rate" in table_data[row-1][0] and match_rate == 100.0:
                cell.set_facecolor("#e8f5e9") # Green
            elif "Match Rate" in table_data[row-1][0] and match_rate < 100.0:
                cell.set_facecolor("#ffebee") # Red

plt.title(f"{game_name} Reproducibility Validation", fontsize=14, pad=20, weight="bold")
plt.tight_layout()
plt.show()


---
# Test Dataset 2 (Deadly Corridor)

> [!WARNING]
> **RESTART KERNEL**: 



In [ ]:
# Setup again (since you restarted the kernel)
import os
import json
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import gymnasium as gym
import vizdoom.gymnasium_wrapper

N_REPLAYS = 100
DATASET_2 = os.path.abspath("../data/sub-01_20260909-135403")

def run_replays(game_name, env_kwargs, seed, ref_data, n_replays):
    ref_actions = ref_data["actions"]
    ref_rewards = ref_data["rewards"]
    ref_terms = ref_data["terminated"]
    ref_truncs = ref_data["truncated"]
    ref_gamevars = ref_data["gamevariables"]
    
    # NEW: extract episode IDs and seeds to handle datasets with multiple episodes
    ref_episode_ids = ref_data.get("episode_id", np.zeros(len(ref_actions), dtype=int))
    ref_episode_seeds = ref_data["episode_seeds"]
    
    T = len(ref_actions)
    results = []
    
    print(f"Starting {n_replays} replays of {game_name} ({T} steps each)...")
    env = gym.make(game_name, render_mode="rgb_array", **env_kwargs)
    
    for r in range(n_replays):
        current_ep = -1
        diverged_step = None
        diverged_vars = []
        
        for t in range(T):
            ep_id = int(ref_episode_ids[t])
            # If the recorded step belongs to a new episode, reset the environment
            if ep_id != current_ep:
                obs, info = env.reset(seed=int(ref_episode_seeds[ep_id]))
                current_ep = ep_id
                
            obs, reward, term, trunc, info = env.step(ref_actions[t])
            
            if isinstance(obs, dict) and "gamevariables" in obs:
                gvars = np.asarray(obs["gamevariables"])
            else:
                state = env.unwrapped.game.get_state()
                if state is not None and state.game_variables is not None:
                    gvars = state.game_variables.copy()
                else:
                    gvars = np.zeros_like(ref_gamevars[t])
            
            mismatches = []
            if not np.isclose(reward, ref_rewards[t], atol=1e-5): mismatches.append("reward")
            if bool(term) != bool(ref_terms[t]): mismatches.append("terminated")
            if bool(trunc) != bool(ref_truncs[t]): mismatches.append("truncated")
            if not np.allclose(gvars, ref_gamevars[t], atol=1e-5): mismatches.append("gamevariables")
            
            if len(mismatches) > 0 and diverged_step is None:
                diverged_step = t
                diverged_vars = mismatches
                break
                
        exact = (diverged_step is None)
        results.append({
            "replay_idx": r, "exact": exact, "diverged_step": diverged_step,
            "diverged_vars": diverged_vars, "reward_match": exact or ("reward" not in diverged_vars),
            "gamevars_match": exact or ("gamevariables" not in diverged_vars),
            "term_match": exact or ("terminated" not in diverged_vars),
            "trunc_match": exact or ("truncated" not in diverged_vars)
        })
        
    env.close()
    return pd.DataFrame(results)
    
def inspect_dataset(data_dir):
    print(f"--- Inspecting {os.path.basename(data_dir)} ---")
    manifest_path = os.path.join(data_dir, "manifest.json")
    with open(manifest_path, "r") as f: manifest = json.load(f)
    game_curriculum = next(p for p in manifest["curriculum"] if p.get("type") == "game")
    game_phase = next(p for p in manifest["phases"] if p.get("type") == "game")
    npz_name = game_phase["data_file"]
    npz_path = os.path.join(data_dir, npz_name)
    data = np.load(npz_path, allow_pickle=True)
    return manifest, data, game_curriculum, game_phase

manifest2, data2, curr2, phase2 = inspect_dataset(DATASET_2)
env_kwargs2 = curr2.get("env_kwargs", {})
game_name2 = curr2["game"]
seed2 = int(data2["episode_seeds"][0])
actions2 = data2["actions"]



In [ ]:
# Run Experiment for Dataset 2
df2 = run_replays(game_name2, env_kwargs2, seed2, data2, N_REPLAYS)
print(f"Dataset 2 ({game_name2}): {df2['exact'].sum()} / {N_REPLAYS} fully matched.")



In [ ]:
# Output Table for Dataset 2
total_steps2 = len(actions2)
total_states2 = len(df2) * total_steps2
perfect_runs2 = df2['exact'].sum()
divergent_runs2 = len(df2) - perfect_runs2
match_rate2 = (perfect_runs2 / len(df2)) * 100

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis("off")

table_data = [
    ["Total Replays Executed", f"{len(df2)}"],
    ["Total Timesteps / Run", f"{total_steps2:,}"],
    ["Total States Verified", f"{total_states2:,}"],
    ["Perfect Trajectories", f"{perfect_runs2}"],
    ["Divergent Trajectories", f"{divergent_runs2}"],
    ["Trajectory Match Rate", f"{match_rate2:.1f}%"]
]

table = ax.table(cellText=table_data, colLabels=["Metric", "Result"], cellLoc="center", loc="center", colColours=["#2b5c8f", "#2b5c8f"])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.0, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(color="white", weight="bold")
    elif row > 0 and col == 1:
        if "Perfect Trajectories" in table_data[row-1][0] and perfect_runs2 == len(df2): cell.set_facecolor("#e8f5e9")
        elif "Divergent" in table_data[row-1][0] and divergent_runs2 == 0: cell.set_facecolor("#e8f5e9")
        elif "Divergent" in table_data[row-1][0] and divergent_runs2 > 0: cell.set_facecolor("#ffebee")
        elif "Match Rate" in table_data[row-1][0] and match_rate2 == 100.0: cell.set_facecolor("#e8f5e9")
        elif "Match Rate" in table_data[row-1][0] and match_rate2 < 100.0: cell.set_facecolor("#ffebee")

plt.title(f"{game_name2} Reproducibility Validation", fontsize=14, pad=20, weight="bold")
plt.tight_layout()
plt.show()



### Explicit Reward Verification
This cell explicitly confirms that the sequence of rewards generated during replay perfectly matches the sequence of rewards recorded in the dataset.

In [ ]:
print("=== REWARD MATCH CHECK ===")
expected_rewards = data["rewards"]

# We use the df results we already computed which tracks reward mismatches
if "reward_match" in df.columns:
    all_rewards_match = df["reward_match"].all()
    print(f"Do all rewards perfectly match across all replays? {all_rewards_match}")
    
    if all_rewards_match:
        print("\nSUCCESS: The environment is perfectly deterministic regarding rewards!")
        print(f"Total Expected Episode Reward: {expected_rewards.sum():.2f}")
    else:
        print("\nWARNING: Some replays yielded different rewards than the recorded dataset.")
else:
    print("Could not find reward_match column in dataframe.")
